In [1]:
"""
Consensus feature importance ranking: tree-based vs linear models.

Reproduces top-20 consensus features separately for tree-based models
(Decision Tree, GBM, Random Forest, XGBoost) and linear models
(Logistic Regression, Linear SVM), excluding Naive Bayes, then computes
the intersection and combined (union) set for further scrutiny.

Input: CSV with columns Model, Rank, Feature, Mean_Importance,
       Std_Importance, Fold_Count (one row per model x feature x rank).
"""

import csv
from collections import defaultdict

# ---- Configuration ----
INPUT_CSV = "top_20_features_per_model_nodrms.csv"  # change path as needed
TOP_N = 50  # only consider features ranked <= TOP_N within each model

TREE_MODELS = {"Decision Tree", "GBM", "Random Forest", "XGBoost"}
LINEAR_MODELS = {"Logistic Regression", "Linear SVM"}
# Naive Bayes is intentionally excluded


def load_rankings(path, top_n=TOP_N):
    """Load per-model feature rankings from CSV.

    Returns: dict {feature: [(model, rank), ...]}
    """
    data = defaultdict(list)
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rank = int(row["Rank"])
            if rank <= top_n:
                data[row["Feature"]].append((row["Model"], rank))
    return data


def group_consensus(data, group):
    """Compute consensus ranking for a subset of models.

    For each feature, keep only appearances from models in `group`,
    then rank by (a) number of models in the group that include the
    feature in their top N (descending), and (b) average rank across
    those models (ascending) as a tiebreaker.

    Returns: list of (feature, n_models, avg_rank, appearances)
    """
    rows = []
    for feat, appearances in data.items():
        group_apps = [(m, r) for m, r in appearances if m in group]
        if not group_apps:
            continue
        n_models = len(group_apps)
        avg_rank = sum(r for _, r in group_apps) / n_models
        rows.append((feat, n_models, avg_rank, group_apps))
    rows.sort(key=lambda x: (-x[1], x[2]))
    return rows


def print_ranking(title, ranked, top_n=20):
    print(f"\n=== {title} ===")
    for i, (feat, n_models, avg_rank, apps) in enumerate(ranked[:top_n], 1):
        apps_str = ", ".join(
            f"{m}(r{r})" for m, r in sorted(apps, key=lambda x: x[1])
        )
        print(f"{i}\t{feat}\tmodels={n_models}\tavg_rank={avg_rank:.1f}\t{apps_str}")


def main():
    data = load_rankings(INPUT_CSV, top_n=TOP_N)

    tree_ranked = group_consensus(data, TREE_MODELS)
    linear_ranked = group_consensus(data, LINEAR_MODELS)

    print_ranking("TREE-BASED (DT, GBM, RF, XGBoost) — top 20", tree_ranked)
    print_ranking("LINEAR (LR, SVM) — top 20", linear_ranked)

    tree_set = {f for f, n, a, ap in tree_ranked[:TOP_N]}
    linear_set = {f for f, n, a, ap in linear_ranked[:TOP_N]}

    intersection = sorted(tree_set & linear_set)
    combined = sorted(tree_set | linear_set)  # "union" / pooled / combined set

    print(f"\n=== INTERSECTION ({len(intersection)} features) ===")
    for f in intersection:
        print(f)

    print(f"\n=== COMBINED SET ({len(combined)} features) ===")
    for f in combined:
        print(f)

    # Optional: write combined set with source annotation to CSV
    with open("combined_feature_set.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Feature", "In_Tree_Top20", "In_Linear_Top20", "Source"])
        for feat in combined:
            in_tree = feat in tree_set
            in_linear = feat in linear_set
            if in_tree and in_linear:
                source = "Both"
            elif in_tree:
                source = "Tree-only"
            else:
                source = "Linear-only"
            writer.writerow([feat, in_tree, in_linear, source])

    print(f"\nWrote combined_feature_set.csv with {len(combined)} features.")


if __name__ == "__main__":
    main()


=== TREE-BASED (DT, GBM, RF, XGBoost) — top 20 ===
1	T112V	models=4	avg_rank=1.5	Decision Tree(r1), Random Forest(r1), GBM(r2), XGBoost(r2)
2	S283G	models=4	avg_rank=1.8	GBM(r1), XGBoost(r1), Random Forest(r2), Decision Tree(r3)
3	D278A	models=4	avg_rank=2.8	Decision Tree(r2), GBM(r3), Random Forest(r3), XGBoost(r3)
4	G70E	models=4	avg_rank=6.0	Decision Tree(r4), GBM(r5), XGBoost(r6), Random Forest(r9)
5	A248S	models=4	avg_rank=9.0	Decision Tree(r5), XGBoost(r5), GBM(r7), Random Forest(r19)
6	R224Q	models=4	avg_rank=10.8	Decision Tree(r6), XGBoost(r8), GBM(r9), Random Forest(r20)
7	M275L	models=4	avg_rank=13.5	GBM(r8), XGBoost(r10), Decision Tree(r18), Random Forest(r18)
8	A205S	models=4	avg_rank=18.0	GBM(r6), Decision Tree(r7), XGBoost(r19), Random Forest(r40)
9	T218S	models=4	avg_rank=18.2	Decision Tree(r10), GBM(r10), Random Forest(r23), XGBoost(r30)
10	I204V	models=4	avg_rank=20.0	GBM(r12), Decision Tree(r15), XGBoost(r18), Random Forest(r35)
11	I203M	models=3	avg_rank=5.7	GBM(r4)

In [3]:
mutations = [
    "A205S", "A248S", "A265V", "A91T", "D232K", "D278A", "D288E", "D3L",
    "D41X", "D6E", "E11D", "E198Q", "E287L", "E35Q", "G106X", "G134N",
    "G149R", "G163E", "G193E", "G193R", "G47A", "G4R", "G70E", "G82K",
    "H171S", "I135V", "I141V", "I162X", "I200L", "I200R", "I203M", "I204V",
    "I217L", "I217P", "I220L", "I220N", "I251F", "I267L", "I60A", "I72V",
    "I84M", "K136Q", "K136T", "K215G", "K215Q", "K236P", "L101I", "L102L",
    "L172X", "L241R", "L74K", "M178S", "M275L", "M50L", "P233A", "P233L",
    "P90A", "Q177X", "Q216K", "Q216P", "Q221L", "Q252K", "Q274S", "R166K",
    "R166S", "R224Q", "R224X", "R269K", "R284M", "S119P", "S230L", "S24G",
    "S24H", "S255Q", "S283G", "S39N", "T112V", "T122I", "T124N", "T125A",
    "T206S", "T210S", "T218S", "V150X", "V165L", "V201I", "V225T", "V281M",
    "V31I", "Y83F"
]